# Training Trajectories Analysis

Questo notebook analizza le traiettorie dei pedoni durante il training, confrontando:
- Le 10 istanze parallele
- L'evoluzione temporale (primi cicli vs ultimi cicli)
- Diversi livelli del curriculum

**Struttura dei file:**

##  Import Libraries

In [ ]:
%matplotlib inline

import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from pathlib import Path
from typing import Dict, List, Tuple
import seaborn as sns

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
mpl.rcParams['figure.figsize'] = (12, 8)
mpl.rcParams['figure.dpi'] = 100

print(" Libraries imported successfully!")

##  Configuration

In [ ]:
# ========== CONFIGURA QUESTI PARAMETRI ==========

# Path base delle session di training
BASE_PATH = "output/runs/training/"

# Nome della session da analizzare (lascia None per selezionare la più recente)
SESSION_NAME = None  # es: "session_2025-10-07_23-52-13" oppure None

# Numero di istanze parallele (di solito 10)
NUM_INSTANCES = 10

# Framerate (fps) - da constants.gd: PHYSICS_TICKS_PER_SECONDS / TICKS_BETWEEN_LOG
FRAMERATE = 30  # 60 / 2 = 30 fps

# Per analisi temporale: quanti PEDONI prendere (non più frame!)
EARLY_PEDESTRIANS = 10   # Primi 10 pedoni per istanza
LATE_PEDESTRIANS = 10    # Penultimi 10 pedoni per istanza

# Colori per le istanze (uno per ogni istanza 0-9)
INSTANCE_COLORS = plt.cm.tab10(np.linspace(0, 1, NUM_INSTANCES))

print(" Configuration loaded!")

## Session Discovery

Trova automaticamente le session disponibili e i livelli al loro interno.

In [ ]:
def find_sessions(base_path: str) -> List[str]:
    """Trova tutte le session di training disponibili."""
    if not os.path.exists(base_path):
        print(f" Path non trovato: {base_path}")
        return []

    sessions = []
    for item in os.listdir(base_path):
        item_path = os.path.join(base_path, item)
        if os.path.isdir(item_path) and item.startswith('session_'):
            sessions.append(item)

    return sorted(sessions, reverse=True)  # Più recenti prima

def find_levels(session_path: str) -> List[str]:
    """Trova tutti i livelli in una session."""
    levels = []
    for item in os.listdir(session_path):
        item_path = os.path.join(session_path, item)
        if os.path.isdir(item_path) and item.endswith('Batch'):
            # Rimuovi "Batch" dal nome per avere il nome pulito
            level_name = item[:-5]  # Rimuove "Batch"
            levels.append(level_name)
    return sorted(levels)

# Trova sessions disponibili
available_sessions = find_sessions(BASE_PATH)

if not available_sessions:
    print(" Nessuna session trovata!")
else:
    print(f"\n Session disponibili ({len(available_sessions)}):")
    for i, sess in enumerate(available_sessions, 1):
        print(f"  {i}. {sess}")

    # Seleziona session
    if SESSION_NAME is None:
        selected_session = available_sessions[0]
        print(f"\n Session selezionata automaticamente (più recente): {selected_session}")
    else:
        selected_session = SESSION_NAME
        print(f"\n Session selezionata manualmente: {selected_session}")

    SESSION_PATH = os.path.join(BASE_PATH, selected_session)

    # Trova livelli nella session
    available_levels = find_levels(SESSION_PATH)
    print(f"\n🎮 Livelli trovati ({len(available_levels)}):")
    for i, level in enumerate(available_levels, 1):
        print(f"  {i}. {level}")

##  Data Loading Functions

In [ ]:
def load_trajectories(level_name: str, session_path: str) -> pd.DataFrame:
    """
    Carica le traiettorie di un livello e separa le istanze.

    Returns:
        DataFrame con colonne: id, frame, x, y, z, combined_group, instance_id, collision_group
    """
    file_path = os.path.join(session_path, f"{level_name}Batch", "trajectories.txt")

    if not os.path.exists(file_path):
        print(f" File non trovato: {file_path}")
        return None

    # Leggi il file
    df = pd.read_csv(
        file_path,
        sep=r'\s+',
        comment='#',
        header=None,
        names=['id', 'frame', 'x', 'y', 'z', 'combined_group']
    )

    # Estrai instance_id e collision_group
    df['instance_id'] = df['combined_group'] // 10000
    df['collision_group'] = df['combined_group'] % 10000

    print(f" Caricato {level_name}: {len(df)} righe, {df['frame'].max()+1} frame")
    print(f"   Istanze trovate: {sorted(df['instance_id'].unique())}")

    return df

def split_by_phase(df: pd.DataFrame, early_count: int = 10, late_count: int = 10) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Divide i dati prendendo i primi e penultimi pedoni (per ID).

    EARLY: Primi early_count pedoni per ogni istanza (primi 10 ID)
    LATE: Penultimi late_count pedoni per ogni istanza (escludendo gli ultimi 10)

    Args:
        df: DataFrame con le traiettorie
        early_count: Numero di pedoni da prendere all'inizio (default: 10)
        late_count: Numero di pedoni da prendere alla fine (default: 10)

    Returns:
        (early_df, late_df)
    """
    early_ids = []
    late_ids = []

    # Per ogni istanza, trova i primi e penultimi pedoni
    for instance_id in sorted(df['instance_id'].unique()):
        instance_df = df[df['instance_id'] == instance_id]

        # Ottieni tutti gli ID unici ordinati per apparizione
        # Ordiniamo per frame minimo per ogni ID per avere l'ordine cronologico
        id_first_frame = instance_df.groupby('id')['frame'].min().sort_values()
        all_ids = id_first_frame.index.tolist()

        total_pedestrians = len(all_ids)

        # EARLY: Primi N pedoni
        early_instance_ids = all_ids[:early_count]
        early_ids.extend(early_instance_ids)

        # LATE: Penultimi N pedoni (escludendo gli ultimi N)
        if total_pedestrians > (early_count + late_count * 2):
            # Prendi da -(2*late_count) a -late_count
            late_instance_ids = all_ids[-(2*late_count):-late_count]
        elif total_pedestrians > (early_count + late_count):
            # Se non ci sono abbastanza, prendi gli ultimi disponibili escludendo i primi
            late_instance_ids = all_ids[-(late_count):]
        else:
            # Troppo pochi pedoni, usa gli stessi di early
            late_instance_ids = early_instance_ids

        late_ids.extend(late_instance_ids)

        print(f"   Instance {instance_id}: {total_pedestrians} pedoni totali")
        print(f"      Early IDs: {[str(x)[:10] for x in early_instance_ids[:3]]}... (primi {len(early_instance_ids)})")
        print(f"      Late IDs:  {[str(x)[:10] for x in late_instance_ids[:3]]}... (penultimi {len(late_instance_ids)})")

    # Filtra il dataframe per questi ID
    early_df = df[df['id'].isin(early_ids)].copy()
    late_df = df[df['id'].isin(late_ids)].copy()

    print(f"\n Divisione per ID pedoni:")
    print(f"   Early: {len(early_ids)} pedoni, {len(early_df)} punti totali")
    print(f"   Late:  {len(late_ids)} pedoni, {len(late_df)} punti totali")

    return early_df, late_df

print(" Data loading functions defined!")

##  Analysis Functions

In [ ]:
def compute_statistics(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calcola statistiche per ogni istanza.
    """
    stats = []

    for instance_id in sorted(df['instance_id'].unique()):
        instance_df = df[df['instance_id'] == instance_id]

        # Calcola velocità
        instance_df = instance_df.sort_values('frame')
        instance_df['dx'] = instance_df['x'].diff()
        instance_df['dy'] = instance_df['y'].diff()
        instance_df['distance'] = np.sqrt(instance_df['dx']**2 + instance_df['dy']**2)
        instance_df['speed'] = instance_df['distance'] * FRAMERATE  # m/s

        stats.append({
            'instance_id': instance_id,
            'total_points': len(instance_df),
            'unique_pedestrians': instance_df['id'].nunique(),
            'total_frames': instance_df['frame'].nunique(),
            'avg_speed': instance_df['speed'].mean(),
            'max_speed': instance_df['speed'].max(),
            'total_distance': instance_df['distance'].sum(),
            'x_min': instance_df['x'].min(),
            'x_max': instance_df['x'].max(),
            'y_min': instance_df['y'].min(),
            'y_max': instance_df['y'].max(),
        })

    return pd.DataFrame(stats)

def compute_speed_over_time(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calcola la velocità media per frame per ogni istanza.
    """
    df = df.copy().sort_values(['instance_id', 'id', 'frame'])

    # Calcola velocità per ogni pedone
    df['dx'] = df.groupby(['instance_id', 'id'])['x'].diff()
    df['dy'] = df.groupby(['instance_id', 'id'])['y'].diff()
    df['distance'] = np.sqrt(df['dx']**2 + df['dy']**2)
    df['speed'] = df['distance'] * FRAMERATE

    # Media per frame e istanza
    speed_by_frame = df.groupby(['instance_id', 'frame'])['speed'].mean().reset_index()

    return speed_by_frame

print(" Analysis functions defined!")

##  Plotting Functions

In [ ]:
def plot_trajectories_by_instance(df: pd.DataFrame, level_name: str, phase: str = "All"):
    """
    Plot delle traiettorie con tutte le istanze sovrapposte in un singolo grafico.
    Ogni istanza ha un colore diverso.
    """
    fig, ax = plt.subplots(1, 1, figsize=(12, 10))
    fig.suptitle(f'{level_name} - Traiettorie Tutte le Istanze ({phase})',
                 fontsize=16, fontweight='bold')

    # Plot per ogni istanza con colore diverso
    for instance_id in sorted(df['instance_id'].unique()):
        instance_df = df[df['instance_id'] == instance_id]
        color = INSTANCE_COLORS[int(instance_id)]

        # Plot traiettoria per ogni pedone di questa istanza
        for ped_id in instance_df['id'].unique():
            ped_df = instance_df[instance_df['id'] == ped_id].sort_values('frame')
            ax.plot(ped_df['x'], ped_df['y'],
                   alpha=0.6, linewidth=1.5, color=color)

        # Start point per questa istanza
        first_frame = instance_df[instance_df['frame'] == instance_df['frame'].min()]
        ax.scatter(first_frame['x'], first_frame['y'],
                  c=[color], s=150, marker='o',
                  edgecolors='black', linewidths=2,
                  label=f'Inst {instance_id} Start', zorder=5)

        # End point per questa istanza
        last_frame = instance_df[instance_df['frame'] == instance_df['frame'].max()]
        ax.scatter(last_frame['x'], last_frame['y'],
                  c=[color], s=150, marker='X',
                  edgecolors='black', linewidths=2, zorder=5)

    ax.set_xlabel('X (m)', fontsize=12)
    ax.set_ylabel('Y (m)', fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)

    plt.tight_layout()
    plt.show()

def plot_speed_comparison(early_df: pd.DataFrame, late_df: pd.DataFrame, level_name: str):
    """
    Confronta la velocità media tra primi e ultimi cicli.
    """
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f'{level_name} - Confronto Velocità: Primi vs Ultimi Cicli',
                fontsize=16, fontweight='bold')

    # Calcola velocità
    early_speed = compute_speed_over_time(early_df)
    late_speed = compute_speed_over_time(late_df)

    # Plot 1: Speed over time per istanza - Early
    ax = axes[0]
    for instance_id in sorted(early_speed['instance_id'].unique()):
        instance_speed = early_speed[early_speed['instance_id'] == instance_id]
        ax.plot(instance_speed['frame'], instance_speed['speed'],
               label=f'Inst {instance_id}', alpha=0.7, linewidth=2,
               color=INSTANCE_COLORS[int(instance_id)])
    ax.set_title('Primi Cicli (Early Training)', fontsize=14)
    ax.set_xlabel('Frame', fontsize=12)
    ax.set_ylabel('Velocità Media (m/s)', fontsize=12)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(True, alpha=0.3)

    # Plot 2: Speed over time per istanza - Late
    ax = axes[1]
    for instance_id in sorted(late_speed['instance_id'].unique()):
        instance_speed = late_speed[late_speed['instance_id'] == instance_id]
        ax.plot(instance_speed['frame'], instance_speed['speed'],
               label=f'Inst {instance_id}', alpha=0.7, linewidth=2,
               color=INSTANCE_COLORS[int(instance_id)])
    ax.set_title('Ultimi Cicli (Late Training)', fontsize=14)
    ax.set_xlabel('Frame', fontsize=12)
    ax.set_ylabel('Velocità Media (m/s)', fontsize=12)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

def plot_statistics_comparison(early_stats: pd.DataFrame, late_stats: pd.DataFrame, level_name: str):
    """
    Confronta le statistiche tra primi e ultimi cicli.
    """
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle(f'{level_name} - Confronto Statistiche per Istanza',
                fontsize=16, fontweight='bold')

    x = np.arange(NUM_INSTANCES)
    width = 0.35

    # Plot 1: Average Speed
    ax = axes[0, 0]
    ax.bar(x - width/2, early_stats['avg_speed'], width, label='Early', alpha=0.8)
    ax.bar(x + width/2, late_stats['avg_speed'], width, label='Late', alpha=0.8)
    ax.set_xlabel('Instance ID')
    ax.set_ylabel('Velocità Media (m/s)')
    ax.set_title('Velocità Media')
    ax.set_xticks(x)
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Plot 2: Total Distance
    ax = axes[0, 1]
    ax.bar(x - width/2, early_stats['total_distance'], width, label='Early', alpha=0.8)
    ax.bar(x + width/2, late_stats['total_distance'], width, label='Late', alpha=0.8)
    ax.set_xlabel('Instance ID')
    ax.set_ylabel('Distanza Totale (m)')
    ax.set_title('Distanza Totale Percorsa')
    ax.set_xticks(x)
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Plot 3: Max Speed
    ax = axes[1, 0]
    ax.bar(x - width/2, early_stats['max_speed'], width, label='Early', alpha=0.8)
    ax.bar(x + width/2, late_stats['max_speed'], width, label='Late', alpha=0.8)
    ax.set_xlabel('Instance ID')
    ax.set_ylabel('Velocità Massima (m/s)')
    ax.set_title('Velocità Massima')
    ax.set_xticks(x)
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Plot 4: Total Points
    ax = axes[1, 1]
    ax.bar(x - width/2, early_stats['total_points'], width, label='Early', alpha=0.8)
    ax.bar(x + width/2, late_stats['total_points'], width, label='Late', alpha=0.8)
    ax.set_xlabel('Instance ID')
    ax.set_ylabel('Numero di Punti')
    ax.set_title('Dati Raccolti')
    ax.set_xticks(x)
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

print(" Plotting functions defined!")

##  Analysis: All Levels Summary

Confronto rapido tra tutti i livelli.

In [ ]:
def analyze_all_levels(levels: List[str], session_path: str):
    """
    Analizza tutti i livelli e crea un summary.
    """
    summary = []

    for level in levels:
        print(f"\n Analyzing {level}...")
        df = load_trajectories(level, session_path)

        if df is None:
            continue

        early_df, late_df = split_by_phase(df, EARLY_PEDESTRIANS, LATE_PEDESTRIANS)

        # Statistiche aggregate
        summary.append({
            'Level': level,
            'Total_Frames': df['frame'].nunique(),
            'Total_Points': len(df),
            'Instances': df['instance_id'].nunique(),
            'Early_Avg_Speed': compute_speed_over_time(early_df)['speed'].mean(),
            'Late_Avg_Speed': compute_speed_over_time(late_df)['speed'].mean(),
            'Speed_Improvement': (
                compute_speed_over_time(late_df)['speed'].mean() -
                compute_speed_over_time(early_df)['speed'].mean()
            ),
        })

    return pd.DataFrame(summary)

# Analizza tutti i livelli
print("\n" + "="*60)
print(" ANALISI DI TUTTI I LIVELLI")
print("="*60)

summary_df = analyze_all_levels(available_levels, SESSION_PATH)

print("\n SUMMARY TABLE:")
print(summary_df.to_string(index=False))

###  Plot: Confronto tra Livelli

In [ ]:
# Seleziona quali livelli visualizzare
LEVELS_TO_PLOT = ["labyrinth"]  # Aggiungi i nomi che vuoi

for level in LEVELS_TO_PLOT:
    if level not in available_levels:
        print(f" {level} non trovato, saltato")
        continue

    # Carica dati
    df = load_trajectories(level, SESSION_PATH)

    if df is None:
        print(f" Saltato {level} - file non trovato")
        continue

    # Dividi in early e late
    early_df, late_df = split_by_phase(df, EARLY_PEDESTRIANS, LATE_PEDESTRIANS)

    print(f"\n Dati caricati:")
    print(f"   - Early pedoni: {len(early_df)} righe ({early_df['id'].nunique()} pedoni)")
    print(f"   - Late pedoni:  {len(late_df)} righe ({late_df['id'].nunique()} pedoni)")

    # GRAFICO 1: Traiettorie Early
    print(f"\n Primi Cicli (Early Training):")
    plot_trajectories_by_instance(early_df, level, "Early Cycles")

    # GRAFICO 2: Traiettorie Late
    print(f"\n Ultimi Cicli (Late Training):")
    plot_trajectories_by_instance(late_df, level, "Late Cycles")



##  Export Results

Esporta i risultati dell'analisi.

In [ ]:
# Esporta summary come CSV
if len(summary_df) > 0:
    output_path = os.path.join(SESSION_PATH, "analysis_summary.csv")
    summary_df.to_csv(output_path, index=False)
    print(f"\n Summary esportato in: {output_path}")

# Esporta statistiche dettagliate del livello analizzato
if df is not None:
    detailed_output_path = os.path.join(SESSION_PATH, f"{LEVEL_TO_ANALYZE}_detailed_stats.csv")

    # Combina early e late stats
    early_stats['phase'] = 'early'
    late_stats['phase'] = 'late'
    detailed_stats = pd.concat([early_stats, late_stats])

    detailed_stats.to_csv(detailed_output_path, index=False)
    print(f" Statistiche dettagliate di {LEVEL_TO_ANALYZE} esportate in: {detailed_output_path}")